# Strands interventions, suspended decisions and agent keys

Five things a Highflame agent can do from SDK 0.6.0, on a [Strands](https://strandsagents.com)
agent running on Amazon Bedrock:

1. **`HighflameIntervention`.** Strands' native interventions API checks each prompt, tool call,
   tool result and model reply. Refused content never reaches the model or the history.
2. **`defer`, returned as a decision.** A policy can suspend an action instead of allowing or
   denying it. The SDK hands that back as a value, and the intervention blocks on it.
3. **`step_up`.** The same, for an action that needs a person to approve it. Explained here; it
   does not run (see section 3 for why).
4. **`agents.update(public_key_pem=...)`.** An admin gives a delegation key to an agent that was
   registered or adopted without one.
5. **Act for a person** with `tokens.exchange_id_token` and `tokens.redeem_id_jag`. The cells
   skip on the hosted service, and say why.

> New to these notebooks? Start with
> [`strands_bedrock_agent_identity.ipynb`](strands_bedrock_agent_identity.ipynb). This notebook
> uses the same Studio identity and setup.

## Setup

### 1. The identity, the guardrail policy and the allow-list

Use the identity you registered for
[`strands_bedrock_agent_identity.ipynb`](strands_bedrock_agent_identity.ipynb), with its setup
steps 1 to 4:

| What | Where it is set up |
| --- | --- |
| An `agent` identity, sub type `orchestrator`, whose credential policy grants `nhi:manage`, `tools:read`, `tools:execute`, `orders:read`, `kb:read` | setup step 1 of that notebook |
| **Structural PII** deployed in enforce mode | its setup step 3 |
| An allow-list granting the tools `lookup_order` and `search_kb` | its setup step 4 |

`nhi:manage` is what lets the key register the throwaway agents below and set their keys.

### 2. The `defer` policy

Section 2 needs one Cedar policy. It defers prompts only from agents whose `publisher` is
`highflame-cookbook-defer-demo`. The notebook registers such an agent and deletes it at the end,
so nothing else in your project is deferred.

**Studio → Custom Agents → Policies → Create Policy**, switch to the raw Cedar view, name it
`cookbook-defer-demo`, set the mode to **Enforce**, paste this and deploy:

```cedar
@defer_on_conflict("")
permit (
    principal,
    action == Guardrails::Action::"process_prompt",
    resource
)
when {
    context has principal &&
    context.principal has publisher &&
    context.principal.publisher == "highflame-cookbook-defer-demo"
};
```

`@defer_on_conflict` turns the permit's `allow` into `defer`. Skip this step and section 2 says
so and moves on.

`publisher` is set by whoever registers the agent, so it is fine for scoping a demo, where the
policy can only make things stricter. Do not use a field the registrant chooses to grant access.

### 3. AWS

Credentials that can call a Bedrock model: `AWS_PROFILE` or the default credential chain, as in
the Strands notebook. `BEDROCK_MODEL_ID` is optional.

In [1]:
%pip install -q -r requirements.txt

from importlib.metadata import version

from highflame.zeroid.resources.tokens import TokensResource

try:
    import strands.interventions  # noqa: F401  (HighflameIntervention needs Strands >= 1.43)
except ImportError as exc:
    raise RuntimeError(
        f"strands-agents {version('strands-agents')} has no interventions API ({exc}). Re-run the install"
        " cell above, then restart the kernel."
    ) from exc

if not hasattr(TokensResource, "exchange_id_token"):
    raise RuntimeError(
        f"highflame {version('highflame')} has no tokens.exchange_id_token(). This notebook needs"
        " highflame 0.6.0 or later. Re-run the install cell above, read its output, then restart the kernel."
    )
print("highflame     ", version("highflame"))
print("strands-agents", version("strands-agents"))

Note: you may need to restart the kernel to use updated packages.


highflame      0.6.0
strands-agents 1.61.1


In [2]:
import getpass
import logging
import os
import uuid

import boto3
from botocore.exceptions import BotoCoreError, ClientError
from dotenv import load_dotenv

from highflame import Highflame
from highflame.integrations.strands import HighflameIntervention
from highflame.zeroid import generate_keypair
from highflame.zeroid.assertions import build_actor_assertion
from highflame.zeroid.errors import APIError
from strands import Agent, tool
from strands.models import BedrockModel

load_dotenv()  # .env beside this notebook; real environment variables win

# Prompted for rather than printed, so the key never lands in the notebook's saved output.
HIGHFLAME_API_KEY = os.environ.get("HIGHFLAME_API_KEY") or getpass.getpass(
    "Orchestrator API key from Studio (input hidden): "
).strip()
BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID")  # None = Strands' default Bedrock model

RUN_ID = uuid.uuid4().hex[:6]  # every identity name carries it, so re-runs never collide
CREATED: list[tuple[str, str]] = []  # (id, external_id) of each agent this run registers, for the clean-up cell

# Which deployment to talk to. Unset = the hosted service. Set both for your own deployment.
ENDPOINTS: dict[str, str] = {}
if os.environ.get("HIGHFLAME_BASE_URL"):
    ENDPOINTS["base_url"] = os.environ["HIGHFLAME_BASE_URL"]
if os.environ.get("HIGHFLAME_IDENTITY_URL"):
    identity_url = os.environ["HIGHFLAME_IDENTITY_URL"].rstrip("/")
    ENDPOINTS["identity_base_url"] = identity_url
    ENDPOINTS["token_url"] = os.environ.get("HIGHFLAME_TOKEN_URL") or f"{identity_url}/oauth2/token"
HOSTED = "identity_base_url" not in ENDPOINTS  # section 5 reads this


def highflame_client(**credential: str) -> Highflame:
    return Highflame(**credential, **ENDPOINTS)


AWS_SESSION = boto3.Session()  # AWS_PROFILE or the default credential chain


def bedrock_model() -> BedrockModel:
    return BedrockModel(boto_session=AWS_SESSION, **({"model_id": BEDROCK_MODEL_ID} if BEDROCK_MODEL_ID else {}))


admin = highflame_client(api_key=HIGHFLAME_API_KEY)
ISSUER = admin.whoami()["issuer"]  # the audience of every assertion signed below. Also proves the key works.

# One model call before anything is registered, so bad AWS credentials fail here.
try:
    await Agent(model=bedrock_model(), callback_handler=None).invoke_async("Reply with the single word OK.")
except (BotoCoreError, ClientError) as exc:
    code = exc.response["Error"]["Code"] if isinstance(exc, ClientError) else type(exc).__name__
    raise RuntimeError(f"Bedrock refused a one-word call ({code}). Refresh your AWS credentials or set BEDROCK_MODEL_ID.") from exc

print("connected to:", "hosted Highflame" if HOSTED else ENDPOINTS["identity_base_url"])

connected to: hosted Highflame


## 1. `HighflameIntervention`

`HighflameIntervention` is a Strands
[intervention handler](https://strandsagents.com): you pass it as `Agent(interventions=[...])`. It
needs strands-agents 1.43 or later. On an older Strands, use `HighflameStrandsHooks`, as the other
Strands notebooks do.

| Checked | `deny`, `step_up` or `defer` | `modify` |
| --- | --- | --- |
| the prompt | replaced before Strands adds it to the history; the model is not called | redacted, when it is a single text block |
| a tool call | denied; the tool body does not run | blocked |
| a tool result | replaced with `[Blocked by Highflame: <reason>]` | redacted, or blocked |
| the model's reply | replaced with the refusal; its tool calls do not run | redacted, or blocked |

Unlike the hooks, a refusal does not raise: the agent returns normally and the refusal is the
answer. So the cells below print the answer and the list of tool bodies that ran.

When Highflame cannot be reached (a connection error, 429 or 5xx), the action is blocked. Pass
`fail_open=True` to let it through instead, knowing that during an outage content Highflame would
refuse then passes unchecked. A rejected key is raised, never passed.

In [3]:
ORDERS = {
    "1042": {"status": "shipped", "carrier": "UPS", "eta": "2 days"},
    # A record a back-office system should never have put in an order: a full card number.
    "2077": {"status": "refund pending", "card_on_file": "4111 1111 1111 1111"},
}
TOOL_CALLS: list[str] = []


@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by its ID and return what the order system holds."""
    TOOL_CALLS.append(f"lookup_order({order_id})")
    return str(ORDERS.get(order_id, "no such order"))


@tool
def search_kb(query: str) -> str:
    """Search the support knowledge base for policies and how-tos."""
    TOOL_CALLS.append("search_kb")
    return "Refunds are accepted within 30 days of delivery."


def build_agent(client: Highflame) -> Agent:
    """A fresh Strands agent on Bedrock, checked by Highflame as `client`'s identity."""
    return Agent(
        model=bedrock_model(),
        system_prompt="You are a customer-support agent. Use your tools to answer.",
        tools=[lookup_order, search_kb],
        interventions=[HighflameIntervention(client)],  # mode="enforce" is the default
        callback_handler=None,  # print the answer once, below, instead of streaming it
    )


async def ask(client: Highflame, prompt: str, session: str) -> str:
    TOOL_CALLS.clear()
    result = await build_agent(client).invoke_async(prompt, invocation_state={"session_id": f"{session}-{RUN_ID}"})
    answer = str(result).strip()
    print("answer    :", answer[:300])
    print("tools ran :", TOOL_CALLS or "none")
    return answer

An ordinary question. The tool runs and the model answers.

In [4]:
await ask(admin, "What's the status of order 1042?", "intervention-ok");

answer    : Your order **#1042** has been **shipped**! Here are the details:

- **Carrier:** UPS
- **Estimated Arrival:** 2 days

Is there anything else I can help you with?
tools ran : ['lookup_order(1042)']


A prompt with a card number. Structural PII refuses it before the model is called, so the model
never sees the number and the refusal is what Strands keeps in the history.

In [5]:
answer = await ask(admin, "My card is 4111 1111 1111 1111, please save it for next time.", "intervention-prompt")
assert "Blocked by Highflame" in answer or "DENIED" in answer, "the prompt was not refused: is Structural PII deployed in enforce mode?"

answer    : DENIED: [Blocked by Highflame: Enterprise Policies Triggered: Structural PII]
tools ran : none


A tool result with a card number. The tool runs, because the call itself is allowed, but the
result is checked before the model reads it. Which check refuses can vary from run to run: when
it is the check on the tool result, the model never sees the card number; when it is the check on
the model's reply, the reply is replaced. Either way the card number never reaches the answer.

In [6]:
answer = await ask(admin, "Look up order 2077 and tell me everything the lookup returns, including the card on file.", "intervention-result")
assert "4111" not in answer, "the card number reached the answer"
assert TOOL_CALLS, "the tool never ran, so its result was not checked; re-run the cell"
print("\nThe card number never reached the answer.")

answer    : I'm sorry, but the lookup for order 2077 was **blocked by our system's enterprise data protection policies** due to the presence of **Structural PII (Personally Identifiable Information)**. This means the order record contains sensitive personal data that cannot be fully retrieved or displayed in th
tools ran : ['lookup_order(2077)']

The card number never reached the answer.


## 2. `defer`: a decision, not an error

Highflame has five decisions. `allow` and `modify` proceed, `deny` blocks, and `step_up` and
`defer` **suspend** the action: it waits for a person to approve it (`step_up`) or for more
context (`defer`).

Shield sends `defer` as HTTP 425 with the decision in the body. Before SDK 0.6.0 the SDK raised an
error for that status. From 0.6.0 `guard.evaluate*` returns it as a normal response, so the
decision is a value you branch on:

```python
resp = client.guard.evaluate_prompt(text)
if resp.allowed:        # allow or modify
    ...
elif resp.suspended:    # step_up or defer: hold the action
    ...
else:                   # deny
    ...
```

**Gate on `resp.allowed`, not on `resp.denied`.** Code that only stops on `deny` now lets `defer`
and `step_up` through.

The cell registers a throwaway agent whose `publisher` the policy from setup step 2 matches, and
checks a harmless prompt as that agent.

In [7]:
DEFER_PUBLISHER = "highflame-cookbook-defer-demo"

held = admin.agents.register(
    name=f"Defer demo {RUN_ID}",
    external_id=f"defer-demo-{RUN_ID}",
    identity_type="agent",
    sub_type="tool_agent",
    trust_level="first_party",
    publisher=DEFER_PUBLISHER,  # what the Cedar policy matches
    description="cookbook defer demo, safe to delete",
)
CREATED.append((held.agent.id, held.agent.external_id))
held_client = highflame_client(api_key=held.api_key)

resp = held_client.guard.evaluate_prompt("What's the status of order 1042?", session_id=f"defer-{RUN_ID}")
print("decision  :", resp.decision)
print("allowed   :", resp.allowed)
print("suspended :", resp.suspended)
print("denied    :", resp.denied)
print("reason    :", resp.policy_reason)

DEFER_DEPLOYED = resp.decision == "defer"
if not DEFER_DEPLOYED:
    print(
        f"\nSKIPPED: the decision is {resp.decision!r}, not 'defer'. Deploy the policy from setup step 2"
        " (Studio, Custom Agents, Policies), wait a minute for it to reach Shield, and re-run this cell."
    )

decision  : defer
allowed   : False
suspended : True
denied    : False
reason    : Policy "cookbook-defer-demo" matched with @defer_on_conflict; operator must resolve the conflict before this action can proceed.


The same agent under `HighflameIntervention`. `defer` blocks the prompt exactly as `deny` does:
the model is not called and no tool runs.

In [8]:
if DEFER_DEPLOYED:
    answer = await ask(held_client, "What's the status of order 1042?", "defer-agent")
    assert not TOOL_CALLS, "a tool ran on a deferred prompt"
    assert "cookbook-defer-demo" in answer, "the prompt was not refused by the defer policy"
else:
    print("SKIPPED: the defer policy is not deployed (see the cell above).")

answer    : DENIED: [Blocked by Highflame: Policy "cookbook-defer-demo" matched with @defer_on_conflict; operator must resolve the conflict before this action can proceed.]
tools ran : none


## 3. `step_up`: explained, not run

`step_up` asks a person to approve one action. A permit carrying `@step_up_required` makes Shield
open an approval request (OpenID CIBA) and answer HTTP 401 with `decision: "step_up"`, the
request's `auth_req_id` and a `binding_message` for the approver. The SDK returns that as a
response with `resp.suspended == True`, and `HighflameIntervention` blocks the action.

What this notebook cannot show yet:

* The SDK does not yet wait for the approval and retry the action (the CIBA poll is a planned
  follow-up), so an agent can only stop and report.
* A deployment whose Shield has no approval channel to Highflame's identity service turns a
  `step_up` policy into `deny`, with the reason `step_up_unconfigured`.

So the cell below does not deploy a policy or call Shield. It prints the branch your code takes.

In [9]:
print(
    "SKIPPED: step_up needs an approver and the SDK's approval polling, which is not released yet.\n"
    "Handle it as a suspended decision:\n"
    "    if resp.decision == 'step_up':  # resp.suspended is True, resp.allowed is False\n"
    "        hold the action and tell the user an approval is pending"
)

SKIPPED: step_up needs an approver and the SDK's approval polling, which is not released yet.
Handle it as a suspended decision:
    if resp.decision == 'step_up':  # resp.suspended is True, resp.allowed is False
        hold the action and tell the user an approval is pending


## 4. Give an agent a delegation key: `agents.update(public_key_pem=...)`

An agent's **delegation key** is the public half of a key pair the agent holds. It signs its own
assertions with the private half: to receive delegated authority, to authenticate without an API
key (the jwt-bearer grant), and to act for a person (section 5).

An agent registered without one, or adopted from discovery, cannot do any of that. From SDK 0.6.0
an admin whose key has `nhi:manage` sets the key with `agents.update(public_key_pem=...)`. The
admin never needs the private key: the agent generates the pair and hands over the public half.
In this demo both sides run in one kernel. In production, generate the pair on the agent.

`agents.update` does not prove that the agent holds the private key, so a key with `nhi:manage`
can give any agent in its project a key it controls itself. Grant that scope narrowly. An agent
that already has a credential of its own should use `agents.enroll_key` instead, which proves it.

In [10]:
keyless = admin.agents.register(
    name=f"Keyless agent {RUN_ID}",
    external_id=f"keyless-{RUN_ID}",
    identity_type="agent",
    sub_type="tool_agent",
    trust_level="first_party",
    description="cookbook key demo, safe to delete",
)
CREATED.append((keyless.agent.id, keyless.agent.external_id))

# On the agent's side: a new ES256 key pair. Only the public half leaves the agent.
AGENT_PRIVATE_KEY, AGENT_PUBLIC_KEY = generate_keypair()


def sign_in_with_key() -> str:
    """The agent authenticates with its own key (jwt-bearer), no API key involved."""
    assertion = build_actor_assertion(wimse_uri=keyless.agent.wimse_uri, private_key_pem=AGENT_PRIVATE_KEY, audience=ISSUER)
    try:
        token = admin.tokens.issue_jwt_bearer(assertion)
    except APIError as exc:
        return f"refused: {str(exc).split(' for identity ')[0]}"
    return f"token issued for {admin.tokens.verify(token.access_token).external_id}"


before = sign_in_with_key()
print("before the update:", before)
admin.agents.update(keyless.agent.id, public_key_pem=AGENT_PUBLIC_KEY)  # the admin's call
after = sign_in_with_key()
print("after the update :", after)
assert "no public key registered" in before, "expected a refusal for an agent with no key"
assert after.startswith("token issued"), "the agent could not sign in with the key the admin set"

before the update: refused: [400] invalid_grant: no public key registered


after the update : token issued for keyless-4c800e


## 5. Act for a person: `exchange_id_token` and `redeem_id_jag`

A person signs in to your agent at their company's identity provider (Okta or Entra). The agent
then asks Highflame for a token **for that person**, with itself as the actor: `sub` is the
person's Highflame user and `act` names the agent. Two standard ways:

| Call | The agent sends | Standard |
| --- | --- | --- |
| `tokens.exchange_id_token` | the person's ID token from its own sign-in app | RFC 8693 token exchange |
| `tokens.redeem_id_jag` | an ID-JAG that Okta Cross App Access issued for the person | RFC 7523 jwt-bearer |

Both authenticate with the agent's own key (section 4); the agent holds no Highflame secret. Both
need a one-time setup: the identity provider added and verified as a trusted issuer, the agent
linked to its app's client id there, and the person linked to a member of your organization.

**On the hosted service the cells below skip.** Highflame cannot yet link a person from an
identity provider to a member of a hosted organization, so every person grant there is refused
with `invalid_grant` ("the person is not a member of this organization"). On a customer-managed
deployment, a person is linked by their verified email on a domain the organization has claimed.

A full walkthrough, with the Okta and Studio setup and a sign-in script, is in progress in
[highflame-cookbook#60](https://github.com/highflame-ai/highflame-cookbook/pull/60).

To run these cells on a customer-managed deployment, set in `.env`: `HIGHFLAME_IDENTITY_URL`,
`IDP_CLIENT_ID` (the agent's app at the identity provider), `IDP_AGENT_PRIVATE_KEY_FILE` (the
private key of an agent already linked to that app), and `PERSON_ID_TOKEN` or `PERSON_ID_JAG`.
Those two are bearer credentials for the person: keep them out of git, and clear this section's
output before you commit or share the notebook, since it prints the person's user id.

In [11]:
import pathlib

IDP_CLIENT_ID = os.environ.get("IDP_CLIENT_ID", "")
KEY_FILE = os.environ.get("IDP_AGENT_PRIVATE_KEY_FILE", "")
PERSON_ID_TOKEN = os.environ.get("PERSON_ID_TOKEN", "")
PERSON_ID_JAG = os.environ.get("PERSON_ID_JAG", "")
RESOURCE = os.environ.get("PERSON_RESOURCE", "")  # optional: the resource the token is bound to


def person_flow_skip_reason(grant_input: str, name: str) -> str | None:
    if HOSTED:
        return ("the hosted service cannot link a person to a member of your organization yet, so it "
                "refuses every person grant with invalid_grant. Run this against a customer-managed deployment.")
    missing = [n for n, v in (("IDP_CLIENT_ID", IDP_CLIENT_ID), ("IDP_AGENT_PRIVATE_KEY_FILE", KEY_FILE), (name, grant_input)) if not v]
    return f"set {', '.join(missing)} in .env" if missing else None


def show_person_token(token) -> None:
    claims = admin.tokens.verify(token.access_token)
    print("issued for the person:", claims.human())  # sub, when principal_type is "user"
    print("acting agent         :", claims.current_actor())  # act.sub


reason = person_flow_skip_reason(PERSON_ID_TOKEN, "PERSON_ID_TOKEN")
if reason:
    print("SKIPPED exchange_id_token:", reason)
else:
    show_person_token(admin.tokens.exchange_id_token(
        PERSON_ID_TOKEN,
        client_id=IDP_CLIENT_ID,
        private_key_pem=pathlib.Path(KEY_FILE).read_text(),
        resource=RESOURCE,
    ))

SKIPPED exchange_id_token: the hosted service cannot link a person to a member of your organization yet, so it refuses every person grant with invalid_grant. Run this against a customer-managed deployment.


In [12]:
reason = person_flow_skip_reason(PERSON_ID_JAG, "PERSON_ID_JAG")
if reason:
    print("SKIPPED redeem_id_jag:", reason)
else:
    show_person_token(admin.tokens.redeem_id_jag(
        PERSON_ID_JAG,
        client_id=IDP_CLIENT_ID,
        private_key_pem=pathlib.Path(KEY_FILE).read_text(),
        resource=RESOURCE,
    ))

SKIPPED redeem_id_jag: the hosted service cannot link a person to a member of your organization yet, so it refuses every person grant with invalid_grant. Run this against a customer-managed deployment.


## Clean up

Deactivates the agents this run registered. If a cell above failed, run this cell anyway: the
defer demo agent holds a live key until it is deactivated. Leave the `cookbook-defer-demo` policy
in place for the next run, or delete it in Studio; it matches only agents with that publisher.

In [13]:
for agent_id, name in CREATED:
    try:
        admin.agents.delete(agent_id)
        print("deleted", name)
    except APIError as exc:
        print(f"could not delete {name}: {exc}")
CREATED.clear()

deleted defer-demo-4c800e


deleted keyless-4c800e
